In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

print("Synthetic railway data generator started.")

Synthetic railway data generator started.


In [2]:
sections = [
    "NDL-MTJ-01",
    "MTJ-AGC-01",
    "AGC-GWL-01",
    "GWL-JHS-01",
    "JHS-BINA-01",
    "BINA-BPL-01",
    "BPL-RTM-01",
    "RTM-VAD-01",
    "VAD-SRT-01",
    "SRT-MUM-01"
]

print("Number of sections:", len(sections))

Number of sections: 10


In [3]:
asset_types = {
    "ENGINEERING": ["TRACK", "BRIDGE", "TURNOUT"],
    "S&T": ["SIGNAL", "POINT_MACHINE", "AXLE_COUNTER"],
    "TRACTION": ["OHE", "TRANSFORMER", "SECTIONING_POST"]
}

In [4]:
n_assets = 1000

asset_rows = []

for i in range(n_assets):

    department = np.random.choice(
        list(asset_types.keys())
    )

    asset_type = np.random.choice(
        asset_types[department]
    )

    section_id = np.random.choice(sections)

    installation_year = np.random.randint(
        2005,
        2024
    )

    criticality = np.random.randint(
        5,
        11
    )

    asset_rows.append({
        "asset_id": f"ASSET{i+1:05d}",
        "section_id": section_id,
        "department": department,
        "asset_type": asset_type,
        "installation_year": installation_year,
        "criticality": criticality
    })

assets_sim = pd.DataFrame(asset_rows)

print(assets_sim.head())
print("Assets:", len(assets_sim))

     asset_id   section_id department       asset_type  installation_year  \
0  ASSET00001   RTM-VAD-01   TRACTION              OHE               2011   
1  ASSET00002   RTM-VAD-01   TRACTION  SECTIONING_POST               2008   
2  ASSET00003   MTJ-AGC-01        S&T           SIGNAL               2016   
3  ASSET00004  JHS-BINA-01        S&T    POINT_MACHINE               2005   
4  ASSET00005   VAD-SRT-01        S&T    POINT_MACHINE               2021   

   criticality  
0            6  
1            7  
2           10  
3            8  
4            7  
Assets: 1000


In [5]:
snapshot_dates = pd.date_range(
    start="2019-01-01",
    end="2026-08-01",
    freq="MS"
)

print("Number of months:", len(snapshot_dates))


Number of months: 92


In [6]:
state_rows = []

In [7]:
for _, asset in assets_sim.iterrows():

    initial_condition = np.random.uniform(75, 100)

    for date in snapshot_dates:

        age = (
            date.year -
            asset["installation_year"]
        )

        age = max(age, 0)

        degradation = age * np.random.uniform(
            0.3,
            1.0
        )

        noise = np.random.normal(
            0,
            3
        )

        condition = (
            initial_condition
            - degradation
            + noise
        )

        condition = np.clip(
            condition,
            10,
            100
        )

        state_rows.append({
            "asset_id": asset["asset_id"],
            "section_id": asset["section_id"],
            "department": asset["department"],
            "asset_type": asset["asset_type"],
            "snapshot_date": date,
            "asset_age_years": age,
            "condition_score": condition,
            "criticality": asset["criticality"]
        })

In [8]:
historical_states = pd.DataFrame(state_rows)

print(historical_states.shape)
print(historical_states.head())

(92000, 8)
     asset_id  section_id department asset_type snapshot_date  \
0  ASSET00001  RTM-VAD-01   TRACTION        OHE    2019-01-01   
1  ASSET00001  RTM-VAD-01   TRACTION        OHE    2019-02-01   
2  ASSET00001  RTM-VAD-01   TRACTION        OHE    2019-03-01   
3  ASSET00001  RTM-VAD-01   TRACTION        OHE    2019-04-01   
4  ASSET00001  RTM-VAD-01   TRACTION        OHE    2019-05-01   

   asset_age_years  condition_score  criticality  
0                8        93.126212            6  
1                8        92.456850            6  
2                8        95.602455            6  
3                8        96.610857            6  
4                8        94.519420            6  


In [9]:
section_usage = {
    section: np.random.uniform(0.5, 1.5)
    for section in sections
}

In [10]:
historical_states["usage_factor"] = (
    historical_states["section_id"]
    .map(section_usage)
)

In [11]:
print(
    historical_states[
        [
            "section_id",
            "usage_factor"
        ]
    ].drop_duplicates().head()
)

      section_id  usage_factor
0     RTM-VAD-01      0.565548
184   MTJ-AGC-01      0.909819
276  JHS-BINA-01      0.662164
368   VAD-SRT-01      1.141546
460   SRT-MUM-01      1.118228


In [12]:
historical_states["weather_stress"] = np.random.uniform(
    0,
    1,
    len(historical_states)
)

In [13]:
maintenance_rows = []

task_counter = 1

for _, row in historical_states.iterrows():

    maintenance_probability = (
        0.01
        + max(0, 70 - row["condition_score"]) * 0.002
    )

    if np.random.random() < maintenance_probability:

        maintenance_rows.append({
            "task_id": f"TASK{task_counter:06d}",
            "asset_id": row["asset_id"],
            "section_id": row["section_id"],
            "department": row["department"],
            "maintenance_date": row["snapshot_date"],
            "maintenance_type": np.random.choice(
                ["INSPECTION", "PREVENTIVE", "REPAIR"]
            ),
            "severity": np.random.randint(3, 10),
            "duration_hours": np.random.uniform(0.5, 4.0)
        })

        task_counter += 1

In [14]:
maintenance_sim = pd.DataFrame(
    maintenance_rows
)

print(maintenance_sim.shape)
print(maintenance_sim.head())

(957, 8)
      task_id    asset_id  section_id department maintenance_date  \
0  TASK000001  ASSET00002  RTM-VAD-01   TRACTION       2022-01-01   
1  TASK000002  ASSET00002  RTM-VAD-01   TRACTION       2023-06-01   
2  TASK000003  ASSET00003  MTJ-AGC-01        S&T       2019-10-01   
3  TASK000004  ASSET00003  MTJ-AGC-01        S&T       2020-08-01   
4  TASK000005  ASSET00003  MTJ-AGC-01        S&T       2022-02-01   

  maintenance_type  severity  duration_hours  
0       PREVENTIVE         8        2.288423  
1       PREVENTIVE         5        1.277000  
2           REPAIR         3        1.677256  
3           REPAIR         5        2.000026  
4       INSPECTION         3        1.553127  


In [15]:
historical_states["failure_score"] = (
    0.04 * historical_states["asset_age_years"]
    + 0.06 * (100 - historical_states["condition_score"])
    + 1.5 * historical_states["usage_factor"]
    + 1.2 * historical_states["weather_stress"]
    + 0.5 * (
        historical_states["criticality"] / 10
    )
)

In [16]:
historical_states["failure_probability"] = (
    1 /
    (
        1 +
        np.exp(
            -(
                historical_states["failure_score"]
                - 6
            )
        )
    )
)

In [17]:
print(
    historical_states[
        [
            "condition_score",
            "asset_age_years",
            "failure_probability"
        ]
    ].head(20)
)

    condition_score  asset_age_years  failure_probability
0         93.126212                8             0.026091
1         92.456850                8             0.018443
2         95.602455                8             0.039612
3         96.610857                8             0.021084
4         94.519420                8             0.023349
5         94.714184                8             0.030872
6         98.417893                8             0.028791
7         91.194071                8             0.055557
8         92.403716                8             0.030100
9         94.520897                8             0.024298
10        92.391222                8             0.025701
11        93.748515                8             0.047679
12        91.926038                9             0.019066
13        95.975703                9             0.025074
14        93.472344                9             0.018220
15        95.080283                9             0.035123
16        92.8

In [18]:
failure_mask = (
    np.random.random(len(historical_states))
    <
    historical_states["failure_probability"]
    * 0.08
)

In [19]:
failure_states = historical_states[
    failure_mask
].copy()

print(
    "Simulated failures:",
    len(failure_states)
)

Simulated failures: 1048


In [20]:
failures_sim = pd.DataFrame({
    "failure_id": [
        f"FAIL{i+1:06d}"
        for i in range(len(failure_states))
    ],
    "asset_id": failure_states["asset_id"].values,
    "failure_date": failure_states["snapshot_date"].values,
    "failure_type": np.random.choice(
        [
            "TRACK_FAILURE",
            "SIGNAL_FAILURE",
            "OHE_FAILURE",
            "EQUIPMENT_FAILURE"
        ],
        len(failure_states)
    ),
    "severity": np.random.randint(
        5,
        11,
        len(failure_states)
    ),
    "downtime_hours": np.random.uniform(
        1,
        12,
        len(failure_states)
    ),
    "repair_duration_hours": np.random.uniform(
        1,
        8,
        len(failure_states)
    )
})

print(failures_sim.head())

   failure_id    asset_id failure_date    failure_type  severity  \
0  FAIL000001  ASSET00002   2021-12-01     OHE_FAILURE         7   
1  FAIL000002  ASSET00003   2025-01-01  SIGNAL_FAILURE         6   
2  FAIL000003  ASSET00003   2025-05-01     OHE_FAILURE         5   
3  FAIL000004  ASSET00005   2022-01-01   TRACK_FAILURE         8   
4  FAIL000005  ASSET00006   2024-07-01     OHE_FAILURE         9   

   downtime_hours  repair_duration_hours  
0       11.819226               4.491025  
1        9.214649               6.461841  
2        1.167383               6.360502  
3        3.626038               4.412465  
4        2.429383               4.656178  


In [21]:
historical_states = historical_states.drop(
    columns=[
        "failure_score",
        "failure_probability"
    ]
)

In [22]:
assets_sim.to_csv(
    "../data/processed/assets_simulated.csv",
    index=False
)

historical_states.to_csv(
    "../data/processed/asset_history_simulated.csv",
    index=False
)

maintenance_sim.to_csv(
    "../data/processed/maintenance_simulated.csv",
    index=False
)

failures_sim.to_csv(
    "../data/processed/failures_simulated.csv",
    index=False
)

print("✅ Synthetic historical railway data saved.")

✅ Synthetic historical railway data saved.


In [23]:
print("Assets:")
print(assets_sim.shape)

print("\nHistorical states:")
print(historical_states.shape)

print("\nMaintenance:")
print(maintenance_sim.shape)

print("\nFailures:")
print(failures_sim.shape)

Assets:
(1000, 6)

Historical states:
(92000, 10)

Maintenance:
(957, 8)

Failures:
(1048, 7)


In [24]:
print(
    historical_states[
        [
            "asset_id",
            "snapshot_date",
            "condition_score",
            "asset_age_years",
            "criticality",
            "usage_factor",
            "weather_stress"
        ]
    ].head(20)
)

      asset_id snapshot_date  condition_score  asset_age_years  criticality  \
0   ASSET00001    2019-01-01        93.126212                8            6   
1   ASSET00001    2019-02-01        92.456850                8            6   
2   ASSET00001    2019-03-01        95.602455                8            6   
3   ASSET00001    2019-04-01        96.610857                8            6   
4   ASSET00001    2019-05-01        94.519420                8            6   
5   ASSET00001    2019-06-01        94.714184                8            6   
6   ASSET00001    2019-07-01        98.417893                8            6   
7   ASSET00001    2019-08-01        91.194071                8            6   
8   ASSET00001    2019-09-01        92.403716                8            6   
9   ASSET00001    2019-10-01        94.520897                8            6   
10  ASSET00001    2019-11-01        92.391222                8            6   
11  ASSET00001    2019-12-01        93.748515       